# STRIVE Recovery: Rerun Every Invalid Trajectory

This generation-only notebook loads the canonical 1,800-record bundle and reruns
every record that is not publishable. A publishable trajectory must:

- be marked `finished=True`;
- not be timed out;
- stop with `explicit_final_answer`;
- contain a non-empty final answer;
- contain at least one recorded step and one answer step.

Successful replacements are checkpointed immediately. Failed recovery attempts do
not overwrite the source record. The final gate passes only when all 1,800 unique
model-problem trajectories satisfy the publishability predicate.

**Methodology warning:** infrastructure retries repair missing observations.
Repeatedly retrying model-format failures changes the operational benchmark. Keep
the original canonical ZIP for operational reporting and describe the fully
repaired ZIP as a conditional/recovery analysis dataset.


## 1. Setup


In [ ]:
# Generation-only dependencies. No PRM is loaded in this notebook.
%pip install -q "openai>=1.68,<3" "datasets>=3.2,<5"                 "transformers>=4.46,<5" sympy scipy pandas matplotlib seaborn                 tqdm tiktoken


In [ ]:
import ast
import contextlib
import gc
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import tempfile
import threading
import time
import traceback

from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass, field, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import openai
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm

random.seed(42)
np.random.seed(42)
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print(
    f"Runtime versions: Python={sys.version.split()[0]}, openai={openai.__version__}, "
    f"transformers={transformers.__version__}, torch={torch.__version__}, "
    f"CUDA={torch.cuda.is_available()}"
)


## 2. Experiment configuration


In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    name: str
    provider: str
    model_id: str
    api_key_env: str
    rpm: int
    base_url: Optional[str] = None
    request_params: Dict[str, Any] = field(default_factory=dict)
    system_directive: str = ""


@dataclass
class Config:
    protocol_version: str = "strive-react-v11"
    seed: int = 42
    math_count: int = 200
    olympiad_count: int = 100
    smoke_count: int = 5
    # None processes the complete 200+100 set. Set an integer for a balanced smaller paper-run subset.
    paper_sample_limit: Optional[int] = None
    # Fair controller budget shared by every generation model:
    # at most four executed tool actions followed by up to two answer attempts.
    max_steps: int = 6
    max_tool_actions: int = 4
    max_consecutive_format_errors: int = 2
    max_repeated_actions: int = 2
    max_answer_only_violations: int = 1
    # Shared visible+provider output allowance. The v4 smoke showed that 1024 tokens
    # truncated three GPT-OSS actions before any protocol action became visible.
    max_output_tokens_per_step: int = 2048
    temperature: float = 0.0
    request_timeout_sec: int = 90
    request_total_timeout_sec: int = 150
    trajectory_timeout_sec: int = 300
    max_retries: int = 3
    retry_base_sec: float = 2.0
    max_retry_wait_sec: float = 70.0
    rate_limit_cooldown_sec: float = 60.0
    resource_exhausted_cooldown_sec: float = 20.0
    lane_api_error_cooldown_sec: float = 30.0
    lane_breaker_after_failures: int = 3
    lane_breaker_cooldown_sec: float = 120.0
    rpm_utilization: float = 0.90

    sandbox_timeout_sec: int = 20
    sandbox_memory_mb: int = 3072
    max_observation_chars: int = 8000
    max_history_chars: int = 24000

    checkpoint_every_problems: int = 1
    parallel_models: int = 6
    paper_batch_count: int = 3
    retry_infrastructure_failures_on_resume: bool = True
    live_trace: bool = True
    log_api_retries: bool = True
    store_full_model_inputs: bool = True
    require_distinct_nvidia_keys: bool = True
    preflight_probe_endpoints: bool = True
    preflight_probe_tokens: int = 32
    generation_probe_timeout_sec: int = 45
    evaluator_probe_timeout_sec: int = 150
    smoke_min_completion_rate: float = 0.80
    # Recovered truncations remain reported but do not invalidate smoke by themselves.
    smoke_max_truncated_steps_per_model: Optional[int] = None

    # The two PRMs are loaded one at a time. Four-bit loading fits a 16 GB GPU.
    prm_load_in_4bit: bool = True
    prm_max_length: int = 4096
    prm_models: list = field(default_factory=lambda: [
        {
            "key": "math_shepherd_mistral_7b",
            "model_name": "peiyi9979/math-shepherd-mistral-7b-prm",
            "type": "math_shepherd",
        },
        {
            "key": "qwen25_math_prm_7b",
            "model_name": "Qwen/Qwen2.5-Math-PRM-7B",
            "type": "qwen_prm",
            # Pin the reviewed remote-code revision for paper reproducibility.
            "revision": "0610740060112df12585d00a1c5f4624d2f59051",
        },
    ])

    sbert_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    semantic_threshold: float = 0.85
    prm_delta_scale: float = 0.10
    progressive_threshold: float = 0.20
    regressive_threshold: float = -0.20

    # Hybrid step score. Missing critic evidence is neutral (0), not a penalty.
    w_prm: float = 0.45
    w_critic: float = 0.25
    w_tool_gain: float = 0.15
    w_redundancy: float = -0.25
    w_error: float = -0.50

    # Selective LLM review is used only for deterministic-inconclusive correctness and
    # near-boundary non-answer steps. Grounding remains deterministic provenance tracing.
    use_critic_llm: bool = True
    use_correctness_judge_fallback: bool = True
    critic_model_id: str = "google/gemma-4-31b-it"
    # Slot 6 is a dedicated judge credential, independent of every generation lane.
    judge_key_slot: int = 6  # Choose 1..6 explicitly; use 0 for NVIDIA_CRITIC_API_KEY.
    judge_rpm: int = 20

    output_dir: str = field(default_factory=lambda: (
        "/kaggle/working/strive_math_v11"
        if Path("/kaggle/working").exists()
        else str(Path.cwd() / "outputs" / "strive_math_v11")
    ))


cfg = Config()
cfg.output_dir = str(Path(cfg.output_dir).resolve())
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.output_dir, "checkpoints").mkdir(parents=True, exist_ok=True)

REQUIRED_SECRET_NAMES = [
    "NVIDIA_API_KEY_1",
    "NVIDIA_API_KEY_2",
    "NVIDIA_API_KEY_3",
    "NVIDIA_API_KEY_4",
    "NVIDIA_API_KEY_5",
    "OPENAI_API_KEY",
]
OPTIONAL_SECRET_NAMES = ["NVIDIA_API_KEY_6", "NVIDIA_CRITIC_API_KEY"]


def load_api_secrets(secret_names=REQUIRED_SECRET_NAMES):
    """Load Kaggle Secrets into environment variables without printing secret values."""
    loaded = []
    if Path("/kaggle/working").exists():
        try:
            from kaggle_secrets import UserSecretsClient
            secret_client = UserSecretsClient()
            for name in secret_names:
                if os.environ.get(name):
                    loaded.append(name)
                    continue
                try:
                    value = secret_client.get_secret(name)
                    if value:
                        os.environ[name] = value
                        loaded.append(name)
                except Exception:
                    pass
        except Exception:
            pass
    else:
        loaded = [name for name in secret_names if os.environ.get(name)]
    print(f"API secrets available: {len(loaded)}/{len(secret_names)} (values are never displayed)")
    return loaded


# Edit only this block when entering credentials directly in the notebook. Leave a value as
# an empty string to use an existing environment variable or Kaggle Secret with the same name.
# Before sharing or publishing the notebook, clear these values and its saved cell output.
API_KEYS = {
    "NVIDIA_API_KEY_1": "",
    "NVIDIA_API_KEY_2": "",
    "NVIDIA_API_KEY_3": "",
    "NVIDIA_API_KEY_4": "",
    "NVIDIA_API_KEY_5": "",
    "NVIDIA_API_KEY_6": "",
    "OPENAI_API_KEY": "",
    "NVIDIA_CRITIC_API_KEY": "",
}

load_api_secrets(REQUIRED_SECRET_NAMES + OPTIONAL_SECRET_NAMES)
for secret_name, secret_value in API_KEYS.items():
    if str(secret_value).strip():
        os.environ[secret_name] = str(secret_value).strip()
if cfg.judge_key_slot not in {0, 1, 2, 3, 4, 5, 6}:
    raise ValueError("cfg.judge_key_slot must be 0 (dedicated key) or one of 1..6")
configured_count = sum(bool(os.environ.get(name, "").strip()) for name in REQUIRED_SECRET_NAMES)
print(f"API keys configured: {configured_count}/{len(REQUIRED_SECRET_NAMES)}")

# Each NVIDIA model uses a distinct key slot.
MODEL_SPECS = [
    ModelSpec(
        name="ministral-14b",
        provider="nvidia",
        model_id="mistralai/ministral-14b-instruct-2512",
        api_key_env="NVIDIA_API_KEY_1",
        rpm=12,
        base_url="https://integrate.api.nvidia.com/v1",
        request_params={
            "temperature": 0.15,
            "top_p": 1.0,
            "frequency_penalty": 0.0,
            "presence_penalty": 0.0,
        },
    ),
    ModelSpec(
        name="glm-5.2",
        provider="nvidia",
        model_id="z-ai/glm-5.2",
        api_key_env="NVIDIA_API_KEY_2",
        # 6 RPM at 90% utilization gives an 11.1-second minimum request gap.
        rpm=6,
        base_url="https://integrate.api.nvidia.com/v1",
        request_params={"temperature": 0.20, "top_p": 0.95, "seed": 42},
    ),
    ModelSpec(
        name="minimax-m3",
        provider="nvidia",
        model_id="minimaxai/minimax-m3",
        api_key_env="NVIDIA_API_KEY_3",
        rpm=12,
        base_url="https://integrate.api.nvidia.com/v1",
        request_params={
            "temperature": 1.0,
            "top_p": 0.95,
        },
    ),
    ModelSpec(
        name="nemotron-3-nano",
        provider="nvidia",
        model_id="nvidia/nemotron-3-nano-30b-a3b",
        api_key_env="NVIDIA_API_KEY_4",
        rpm=20,
        base_url="https://integrate.api.nvidia.com/v1",
        request_params={
            "temperature": 0.20,
            "top_p": 0.95,
            # Instruct mode avoids spending the whole action budget in reasoning_content.
            "extra_body": {
                "top_k": 1,
                "chat_template_kwargs": {"enable_thinking": False},
            },
        },
    ),
    ModelSpec(
        name="gpt-oss-20b",
        provider="nvidia",
        model_id="openai/gpt-oss-20b",
        api_key_env="NVIDIA_API_KEY_5",
        rpm=12,
        base_url="https://integrate.api.nvidia.com/v1",
        request_params={
            "temperature": 1.0,
            "top_p": 1.0,
            # NVIDIA's real control; the text directive alone did not constrain reasoning_content.
            "reasoning_effort": "low",
        },
        system_directive="Reasoning: low",
    ),
    ModelSpec(
        name="gpt-5-nano",
        provider="openai",
        model_id="gpt-5-nano",
        api_key_env="OPENAI_API_KEY",
        rpm=60,
        request_params={"reasoning_effort": "minimal"},
    ),
]

# Optional explicit standby assignment. This never rotates automatically on HTTP 429.
# Example: {"minimax-m3": "NVIDIA_API_KEY_6"}
NVIDIA_MODEL_KEY_OVERRIDES = {}
allowed_model_key_slots = {f"NVIDIA_API_KEY_{i}" for i in range(1, 7)}
known_model_names = {spec.name for spec in MODEL_SPECS if spec.provider == "nvidia"}
unknown_models = set(NVIDIA_MODEL_KEY_OVERRIDES) - known_model_names
invalid_slots = set(NVIDIA_MODEL_KEY_OVERRIDES.values()) - allowed_model_key_slots
if unknown_models or invalid_slots:
    raise ValueError(
        f"Invalid NVIDIA_MODEL_KEY_OVERRIDES: unknown_models={unknown_models}, "
        f"invalid_key_slots={invalid_slots}"
    )
MODEL_SPECS = [
    replace(spec, api_key_env=NVIDIA_MODEL_KEY_OVERRIDES.get(spec.name, spec.api_key_env))
    for spec in MODEL_SPECS
]

print(f"Output directory: {cfg.output_dir}")
print(f"Paper set: {cfg.math_count} MATH + {cfg.olympiad_count} OlympiadBench")
print(f"Processing limit: {cfg.paper_sample_limit or 'all selected problems'}")
pd.DataFrame([asdict(x) | {"base_url": x.base_url or "OpenAI"} for x in MODEL_SPECS])


## 3. Dataset loading


In [ ]:
def balanced_take(rows: List[Dict], n: int, group_keys: Tuple[str, ...], seed: int) -> List[Dict]:
    """Deterministic round-robin sample across available subject/level groups."""
    rng = random.Random(seed)
    groups = defaultdict(list)
    for row in rows:
        groups[tuple(str(row.get(k, "unknown")) for k in group_keys)].append(row)
    for values in groups.values():
        rng.shuffle(values)
    keys = sorted(groups)
    rng.shuffle(keys)
    chosen = []
    while len(chosen) < n and keys:
        next_keys = []
        for key in keys:
            if groups[key] and len(chosen) < n:
                chosen.append(groups[key].pop())
            if groups[key]:
                next_keys.append(key)
        keys = next_keys
    if len(chosen) != n:
        raise ValueError(f"Requested {n} examples, found only {len(chosen)} eligible rows")
    return chosen


def first_answer(value: Any) -> str:
    if isinstance(value, (list, tuple)):
        return str(value[0]).strip() if value else ""
    return str(value or "").strip()


def load_paper_problem_sets(cfg: Config) -> Tuple[List[Dict], List[Dict]]:
    """Return the frozen paper set and a disjoint pool for smoke/calibration runs."""
    math_ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
    all_math_rows = [
        {
            "id": f"math500:{row.get('unique_id', i)}",
            "dataset": "MATH-500",
            "problem": row["problem"],
            "gold_answer": str(row["answer"]),
            "reference_solution": str(row.get("solution", "")),
            "subject": str(row.get("subject", "unknown")),
            "level": str(row.get("level", "unknown")),
        }
        for i, row in enumerate(math_ds)
    ]
    math_rows = balanced_take(
        all_math_rows,
        cfg.math_count,
        ("subject", "level"),
        cfg.seed,
    )

    olympiad_ds = load_dataset(
        "Hothan/OlympiadBench",
        "OE_TO_maths_en_COMP",
        split="train",
    )
    all_olympiad_rows = []
    for i, row in enumerate(olympiad_ds):
        if row.get("error") not in (None, "", False):
            continue
        if bool(row.get("is_multiple_answer", False)):
            continue
        answer = first_answer(row.get("final_answer"))
        question = str(row.get("question", "")).strip()
        if not question or not answer:
            continue
        all_olympiad_rows.append({
            "id": f"olympiad:{row.get('id', i)}",
            "dataset": "OlympiadBench-OE-TO-maths-en-COMP",
            "problem": question,
            "gold_answer": answer,
            "reference_solution": first_answer(row.get("solution")),
            "subject": str(row.get("subfield", "Olympiad math")),
            "level": str(row.get("difficulty", "Competition")),
        })
    olympiad_rows = balanced_take(
        all_olympiad_rows,
        cfg.olympiad_count,
        ("subject",),
        cfg.seed + 1,
    )
    rows = math_rows + olympiad_rows
    assert len(rows) == cfg.math_count + cfg.olympiad_count
    paper_ids = {row["id"] for row in rows}
    smoke_pool = [
        row for row in all_math_rows + all_olympiad_rows
        if row["id"] not in paper_ids
    ]
    return rows, smoke_pool


def choose_smoke_problems(problems: List[Dict], n: int = 5) -> List[Dict]:
    """Use 3 MATH and 2 OlympiadBench tasks with different subjects when possible."""
    selected, seen = [], set()
    targets = [("MATH-500", 3), ("OlympiadBench", 2)]
    for dataset_prefix, count in targets:
        candidates = [p for p in problems if p["dataset"].startswith(dataset_prefix)]
        for p in candidates:
            key = (p["dataset"], p["subject"])
            if key not in seen:
                selected.append(p)
                seen.add(key)
            if sum(x["dataset"].startswith(dataset_prefix) for x in selected) >= count:
                break
    return selected[:n]


def choose_processing_problems(problems: List[Dict], limit: Optional[int], seed: int) -> List[Dict]:
    if limit is None:
        return list(problems)
    limit = int(limit)
    if limit < 1 or limit > len(problems):
        raise ValueError(f"paper_sample_limit must be between 1 and {len(problems)}, got {limit}")
    return balanced_take(problems, limit, ("dataset", "subject"), seed + 10)


def stratified_problem_batches(
    problems: List[Dict], batch_count: int, seed: int
) -> List[List[Dict]]:
    """Distribute every dataset/subject group round-robin so each batch has similar difficulty."""
    if batch_count < 1:
        raise ValueError("batch_count must be positive")
    rng = random.Random(seed)
    groups = defaultdict(list)
    for problem in problems:
        groups[(problem["dataset"], problem.get("subject", "unknown"))].append(problem)
    batches = [[] for _ in range(batch_count)]
    offset = 0
    for key in sorted(groups):
        values = list(groups[key])
        rng.shuffle(values)
        for index, problem in enumerate(values):
            batches[(offset + index) % batch_count].append(problem)
        offset = (offset + len(values)) % batch_count
    for batch in batches:
        rng.shuffle(batch)
    flattened = [p["id"] for batch in batches for p in batch]
    assert len(flattened) == len(set(flattened)) == len(problems)
    return batches



# Recovery uses problems.jsonl from the canonical generation ZIP.
# No Hugging Face dataset download is required here.


## 4. Stateful multi-step Python sandbox


In [ ]:
SAFE_IMPORT_ROOTS = {
    "math", "cmath", "fractions", "decimal", "statistics", "itertools",
    "collections", "functools", "operator", "sympy", "numpy", "scipy",
}
BLOCKED_CALLS = {
    "open", "exec", "eval", "compile", "input", "breakpoint", "__import__",
    "globals", "locals", "vars", "help", "dir",
}


def validate_math_code(source: str) -> Tuple[bool, str]:
    """Reject filesystem, network, process, and introspection primitives before execution."""
    try:
        tree = ast.parse(source)
    except SyntaxError as exc:
        return False, f"SyntaxError: {exc}"
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split(".")[0] not in SAFE_IMPORT_ROOTS:
                    return False, f"Blocked import: {alias.name}"
        elif isinstance(node, ast.ImportFrom):
            root = (node.module or "").split(".")[0]
            if root not in SAFE_IMPORT_ROOTS:
                return False, f"Blocked import: {node.module}"
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            if node.func.id in BLOCKED_CALLS:
                return False, f"Blocked call: {node.func.id}"
        elif isinstance(node, ast.Attribute) and node.attr.startswith("__"):
            return False, f"Blocked private attribute: {node.attr}"
    return True, "ok"


def capture_final_expression(source: str) -> str:
    """Give subprocess execution Jupyter-like display semantics for a final expression."""
    try:
        tree = ast.parse(source)
        if not tree.body or not isinstance(tree.body[-1], ast.Expr):
            return source
        value = tree.body[-1].value
        already_printed = (
            isinstance(value, ast.Call)
            and isinstance(value.func, ast.Name)
            and value.func.id == "print"
        )
        if already_printed:
            return source
        tree.body[-1] = ast.Expr(
            value=ast.Call(func=ast.Name(id="print", ctx=ast.Load()), args=[value], keywords=[])
        )
        ast.fix_missing_locations(tree)
        return ast.unparse(tree)
    except Exception:
        return source


def truncate_middle(text: str, max_chars: int) -> str:
    text = str(text or "")
    if len(text) <= max_chars:
        return text
    half = max(1, (max_chars - 80) // 2)
    return text[:half] + "\n...[observation truncated]...\n" + text[-half:]


def _apply_resource_limits(memory_mb: int, cpu_seconds: int):
    if os.name != "posix":
        return
    try:
        import resource
        memory = int(memory_mb) * 1024 * 1024
        resource.setrlimit(resource.RLIMIT_AS, (memory, memory))
        resource.setrlimit(resource.RLIMIT_CPU, (cpu_seconds, cpu_seconds + 2))
        resource.setrlimit(resource.RLIMIT_FSIZE, (10 * 1024 * 1024, 10 * 1024 * 1024))
        if hasattr(resource, "RLIMIT_NPROC"):
            resource.setrlimit(resource.RLIMIT_NPROC, (16, 16))
    except Exception:
        pass


SAFE_PREAMBLE = r"""
import math
import cmath
import sympy
import numpy as np
from decimal import Decimal
from fractions import Fraction
from itertools import combinations, permutations, product
from collections import Counter, defaultdict
from sympy import *

np.random.seed(0)
x, y, z, a, b, c, n, k, i, j, t = sympy.symbols("x y z a b c n k i j t")
"""


@dataclass
class SandboxResult:
    ok: bool
    stdout: str
    stderr: str
    returncode: int
    elapsed_sec: float
    timed_out: bool = False
    blocked: bool = False

    def observation(self, max_chars: int) -> str:
        status = "SUCCESS" if self.ok else "FAILURE"
        parts = [f"Execution status: {status}", f"Elapsed: {self.elapsed_sec:.3f}s"]
        if self.stdout.strip():
            parts.append("STDOUT:\n" + self.stdout.strip())
        if self.stderr.strip():
            parts.append("STDERR:\n" + self.stderr.strip())
        if self.timed_out:
            parts.append("TimeoutError: execution exceeded the step limit.")
        if not self.stdout.strip() and not self.stderr.strip() and self.ok:
            parts.append("No printed output. Use print(...) to expose a result.")
        return truncate_middle("\n".join(parts), max_chars)


class MultiStepPythonSandbox:
    """Fresh subprocess per action with successful code replayed silently."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.workdir = Path(tempfile.mkdtemp(prefix="strive_math_"))
        self.successful_blocks: List[str] = []
        self.closed = False

    def _runner_source(self, current: str) -> str:
        history = "\n\n".join(self.successful_blocks)
        return f"""{SAFE_PREAMBLE}
import contextlib
import io
import sys
import traceback

_globals = globals()
_history = {history!r}
_current = {current!r}
try:
    if _history.strip():
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            exec(_history, _globals, _globals)
    exec(_current, _globals, _globals)
except BaseException:
    traceback.print_exc()
    sys.exit(1)
"""

    def run(self, source: str) -> SandboxResult:
        allowed, reason = validate_math_code(source)
        if not allowed:
            return SandboxResult(False, "", reason, 126, 0.0, blocked=True)

        execution_source = capture_final_expression(source)
        script = self.workdir / "runner.py"
        script.write_text(self._runner_source(execution_source), encoding="utf-8")
        clean_env = {
            "PATH": os.environ.get("PATH", ""),
            "PYTHONPATH": "",
            "PYTHONHASHSEED": "0",
            "HOME": str(self.workdir),
            "TMPDIR": str(self.workdir),
            "MPLBACKEND": "Agg",
        }
        start = time.perf_counter()
        try:
            proc = subprocess.run(
                [sys.executable, "-I", str(script)],
                cwd=self.workdir,
                env=clean_env,
                text=True,
                capture_output=True,
                timeout=self.cfg.sandbox_timeout_sec,
                preexec_fn=(
                    lambda: _apply_resource_limits(
                        self.cfg.sandbox_memory_mb,
                        self.cfg.sandbox_timeout_sec,
                    )
                ) if os.name == "posix" else None,
            )
            result = SandboxResult(
                ok=proc.returncode == 0,
                stdout=proc.stdout,
                stderr=proc.stderr,
                returncode=proc.returncode,
                elapsed_sec=time.perf_counter() - start,
            )
        except subprocess.TimeoutExpired as exc:
            result = SandboxResult(
                ok=False,
                stdout=(exc.stdout or "") if isinstance(exc.stdout, str) else "",
                stderr=(exc.stderr or "") if isinstance(exc.stderr, str) else "",
                returncode=124,
                elapsed_sec=time.perf_counter() - start,
                timed_out=True,
            )
        if result.ok:
            self.successful_blocks.append(source)
        return result

    def close(self):
        if not self.closed:
            shutil.rmtree(self.workdir, ignore_errors=True)
            self.closed = True

    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.close()


In [ ]:
# Deterministic infrastructure test: state persists, failed code is not committed,
# stdout is not replayed, and unsafe imports are blocked.
with MultiStepPythonSandbox(cfg) as sb:
    r1 = sb.run("u = 41\nprint('created', u)")
    r2 = sb.run("u += 1\nprint('answer', u)")
    r3 = sb.run("u = 999\nraise ValueError('do not commit this state')")
    r4 = sb.run("print('state_after_failure', u)")
    r5 = sb.run("import os\nprint(os.environ)")
    r6 = sb.run("u * 2")

assert r1.ok and "created 41" in r1.stdout
assert r2.ok and "answer 42" in r2.stdout and "created 41" not in r2.stdout
assert not r3.ok
assert r4.ok and "state_after_failure 42" in r4.stdout
assert r5.blocked and "Blocked import" in r5.stderr
assert r6.ok and r6.stdout.strip() == "84"
print("Sandbox self-test passed: persistence, rollback, muted replay, final-expression display, and policy checks work.")


## 5. API adapters, shared rate limits, and retries


In [ ]:
from openai import APIConnectionError, APITimeoutError, OpenAI


class PacedRateLimiter:
    """Smooth request starts to avoid a burst of 20 calls followed by a 60-second stall."""

    def __init__(self, rpm: int, utilization: float = 0.90):
        self.lock = threading.Lock()
        self.next_start = 0.0
        self.configure(rpm, utilization)

    def configure(self, rpm: int, utilization: float):
        self.rpm = max(1, int(rpm))
        self.utilization = float(np.clip(utilization, 0.1, 1.0))
        self.interval_sec = 60.0 / (self.rpm * self.utilization)

    def acquire(self) -> float:
        with self.lock:
            now = time.monotonic()
            scheduled = max(now, self.next_start)
            self.next_start = scheduled + self.interval_sec
            wait = max(0.0, scheduled - now)
        if wait:
            time.sleep(wait)
        return wait


_clients: Dict[Tuple[str, str, str], OpenAI] = {}
_limiters: Dict[str, PacedRateLimiter] = {}
_registry_lock = threading.Lock()


def resolve_api_key(spec: ModelSpec) -> str:
    key = os.environ.get(spec.api_key_env, "").strip()
    if not key and spec.name == "external-nim-evaluator":
        key = os.environ.get("NVIDIA_API_KEY_5", "").strip()
    if not key and spec.provider == "nvidia" and not cfg.require_distinct_nvidia_keys:
        key = os.environ.get("NVIDIA_API_KEY", "").strip()
    if not key:
        raise RuntimeError(
            f"Missing API key for {spec.name}. Set {spec.api_key_env}"
            + (
                " (distinct NVIDIA key slots are required)"
                if spec.provider == "nvidia" and cfg.require_distinct_nvidia_keys
                else (" or NVIDIA_API_KEY" if spec.provider == "nvidia" else "")
            )
        )
    return key


def client_and_limiter(spec: ModelSpec):
    key = resolve_api_key(spec)
    fingerprint = hashlib.sha256(key.encode()).hexdigest()[:16]
    client_key = (spec.provider, spec.base_url or "OpenAI", fingerprint)
    limiter_key = f"{spec.provider}:{fingerprint}"
    with _registry_lock:
        if client_key not in _clients:
            # Disable SDK-internal retries; the explicit loop below is the sole retry policy.
            kwargs = {
                "api_key": key,
                "timeout": cfg.request_timeout_sec,
                "max_retries": 0,
            }
            if spec.base_url:
                kwargs["base_url"] = spec.base_url
            _clients[client_key] = OpenAI(**kwargs)
        if limiter_key not in _limiters:
            _limiters[limiter_key] = PacedRateLimiter(spec.rpm, cfg.rpm_utilization)
        else:
            # If several models share a key, respect the most conservative configured RPM.
            conservative_rpm = min(_limiters[limiter_key].rpm, spec.rpm)
            _limiters[limiter_key].configure(conservative_rpm, cfg.rpm_utilization)
    return _clients[client_key], _limiters[limiter_key]


def usage_value(obj, name: str, default: int = 0) -> int:
    value = getattr(obj, name, default) if obj is not None else default
    return int(value or 0)


def extract_openai_response_text(response) -> str:
    """Support both the SDK output_text helper and explicit Responses output items."""
    direct = getattr(response, "output_text", "") or ""
    if direct.strip():
        return direct.strip()
    chunks = []
    for item in getattr(response, "output", None) or []:
        for part in getattr(item, "content", None) or []:
            text_value = getattr(part, "text", None)
            if text_value:
                chunks.append(str(text_value))
    return "\n".join(chunks).strip()


class ModelCallError(RuntimeError):
    def __init__(self, message: str, meta: Dict):
        super().__init__(message)
        self.meta = meta


def call_model(
    spec: ModelSpec,
    messages: List[Dict],
    cfg: Config,
    deadline_monotonic: Optional[float] = None,
    context: str = "",
    max_output_tokens_override: Optional[int] = None,
) -> Tuple[str, Dict]:
    client, limiter = client_and_limiter(spec)
    last_error = None
    call_started = time.monotonic()
    call_deadline = call_started + cfg.request_total_timeout_sec
    if deadline_monotonic is not None:
        call_deadline = min(call_deadline, deadline_monotonic)
    rate_wait_total = 0.0
    retry_wait_total = 0.0
    service_time_total = 0.0
    error_history = []
    output_budget = int(max_output_tokens_override or cfg.max_output_tokens_per_step)
    for attempt in range(cfg.max_retries):
        remaining = call_deadline - time.monotonic()
        if remaining <= 1.0:
            last_error = TimeoutError("request retry deadline exhausted")
            break
        rate_wait = limiter.acquire()
        rate_wait_total += rate_wait
        remaining = call_deadline - time.monotonic()
        if remaining <= 1.0:
            last_error = TimeoutError("request deadline exhausted after local RPM pacing")
            break
        attempt_timeout = max(1.0, min(float(cfg.request_timeout_sec), remaining))
        request_client = client.with_options(timeout=attempt_timeout, max_retries=0)
        start = time.perf_counter()
        try:
            if spec.provider == "openai":
                reasoning_effort = spec.request_params.get("reasoning_effort", "low")
                response = request_client.responses.create(
                    model=spec.model_id,
                    input=messages,
                    max_output_tokens=output_budget,
                    reasoning={"effort": reasoning_effort},
                )
                service_latency = time.perf_counter() - start
                service_time_total += service_latency
                text = extract_openai_response_text(response)
                usage = response.usage
                details = getattr(usage, "output_tokens_details", None)
                hidden = usage_value(details, "reasoning_tokens")
                finish_reason = str(getattr(response, "status", "") or "")
                meta = {
                    "input_tokens": usage_value(usage, "input_tokens"),
                    "output_tokens": usage_value(usage, "output_tokens"),
                    "hidden_reasoning_tokens": hidden,
                    "provider_reasoning": "",
                }
            else:
                request_params = dict(spec.request_params)
                configured_max = int(request_params.pop("max_tokens", output_budget))
                max_tokens = min(configured_max, output_budget)
                response = request_client.chat.completions.create(
                    model=spec.model_id,
                    messages=messages,
                    max_tokens=max_tokens,
                    stream=False,
                    **request_params,
                )
                service_latency = time.perf_counter() - start
                service_time_total += service_latency
                message = response.choices[0].message
                text = message.content or ""
                provider_reasoning = (
                    getattr(message, "reasoning", None)
                    or getattr(message, "reasoning_content", None)
                    or ""
                )
                usage = response.usage
                details = getattr(usage, "completion_tokens_details", None)
                hidden = usage_value(details, "reasoning_tokens")
                finish_reason = str(response.choices[0].finish_reason or "")
                meta = {
                    "input_tokens": usage_value(usage, "prompt_tokens"),
                    "output_tokens": usage_value(usage, "completion_tokens"),
                    "hidden_reasoning_tokens": hidden,
                    "provider_reasoning": provider_reasoning,
                }
            meta.update({
                "latency_sec": time.monotonic() - call_started,
                "service_latency_sec": service_latency,
                "service_time_all_attempts_sec": service_time_total,
                "rate_limit_wait_sec": rate_wait_total,
                "retry_wait_sec": retry_wait_total,
                "request_attempts": attempt + 1,
                "retry_errors": error_history,
                "finish_reason": finish_reason,
                "provider_truncated": finish_reason.lower() in {
                    "length", "max_tokens", "incomplete"
                },
            })
            return text.strip(), meta
        except Exception as exc:
            service_time_total += time.perf_counter() - start
            last_error = exc
            status = getattr(exc, "status_code", None)
            retryable = (
                status in {408, 409, 429, 500, 502, 503, 504}
                or isinstance(exc, (APIConnectionError, APITimeoutError))
            )
            error_history.append({
                "attempt": attempt + 1,
                "status": status,
                "type": type(exc).__name__,
                "message": truncate_middle(str(exc), 500),
            })
            if not retryable or attempt + 1 == cfg.max_retries:
                break
            retry_after = getattr(getattr(exc, "response", None), "headers", {}).get("retry-after")
            try:
                delay = float(retry_after)
            except (TypeError, ValueError):
                if status == 429:
                    # A 2-4 second retry loop amplified the observed GLM quota failure.
                    delay = cfg.rate_limit_cooldown_sec + random.random() * 3.0
                elif status == 503 and "ResourceExhausted" in str(exc):
                    delay = cfg.resource_exhausted_cooldown_sec + random.random() * 3.0
                else:
                    delay = cfg.retry_base_sec * (2 ** attempt) + random.random()
            delay = min(float(delay), cfg.max_retry_wait_sec)
            delay = min(delay, max(0.0, call_deadline - time.monotonic() - 1.0))
            if cfg.log_api_retries:
                label = context or spec.name
                tqdm.write(
                    f"[{label}] API retry {attempt + 2}/{cfg.max_retries} after "
                    f"{type(exc).__name__}; waiting {delay:.1f}s"
                )
            if delay > 0:
                time.sleep(delay)
                retry_wait_total += delay
    summary = truncate_middle(str(last_error), 1000)
    failure_meta = {
        "latency_sec": time.monotonic() - call_started,
        "service_latency_sec": 0.0,
        "service_time_all_attempts_sec": service_time_total,
        "rate_limit_wait_sec": rate_wait_total,
        "retry_wait_sec": retry_wait_total,
        "request_attempts": len(error_history),
        "retry_errors": error_history,
        "finish_reason": "error",
        "provider_truncated": False,
        "provider_reasoning": "",
        "input_tokens": 0,
        "output_tokens": 0,
        "hidden_reasoning_tokens": 0,
    }
    raise ModelCallError(
        f"API call failed for {spec.name} after {len(error_history)} failed attempt(s): {summary}",
        failure_meta,
    )


def probe_model_endpoint(spec: ModelSpec, cfg: Config) -> Dict:
    """Make one tiny real request so inaccessible model aliases fail before generation."""
    started = time.monotonic()
    timeout = (
        cfg.evaluator_probe_timeout_sec
        if spec.name == "external-nim-evaluator"
        else cfg.generation_probe_timeout_sec
    )
    try:
        _, meta = call_model(
            spec,
            [{"role": "user", "content": "Reply with exactly: OK"}],
            cfg,
            deadline_monotonic=started + min(float(timeout), cfg.request_total_timeout_sec),
            context=f"preflight:{spec.name}",
            max_output_tokens_override=cfg.preflight_probe_tokens,
        )
        return {
            "endpoint": "reachable",
            "probe_sec": round(float(meta.get("latency_sec", 0.0)), 2),
            "probe_error": "",
        }
    except Exception as exc:
        retry_details = ""
        if isinstance(exc, ModelCallError):
            retry_details = f" retries={exc.meta.get('retry_errors', [])}"
        return {
            "endpoint": "FAILED",
            "probe_sec": round(time.monotonic() - started, 2),
            "probe_error": truncate_middle(str(exc) + retry_details, 900),
        }


def preflight_models(specs: List[ModelSpec], probe_endpoints: Optional[bool] = None):
    probe_endpoints = cfg.preflight_probe_endpoints if probe_endpoints is None else probe_endpoints
    rows = []
    nvidia_fingerprints = []
    for spec in specs:
        try:
            key = resolve_api_key(spec)
            fingerprint = hashlib.sha256(key.encode()).hexdigest()
            if spec.provider == "nvidia":
                nvidia_fingerprints.append((spec.name, fingerprint))
            row = {
                "model": spec.name,
                "provider": spec.provider,
                "key_slot": spec.api_key_env,
                "key": "configured",
                "rpm": spec.rpm,
            }
            rows.append(row)
        except Exception as exc:
            rows.append({
                "model": spec.name,
                "provider": spec.provider,
                "key_slot": spec.api_key_env,
                "key": str(exc),
                "rpm": spec.rpm,
                "endpoint": "not tested",
                "probe_sec": 0.0,
                "probe_error": "",
            })
    frame = pd.DataFrame(rows)
    missing = frame[frame["key"] != "configured"]
    if not missing.empty:
        display(frame)
        raise RuntimeError(
            "Missing API keys. Enter each value once in the API_KEYS block in Section 2, "
            "then rerun Section 2 and this cell. No password prompt is used."
        )
    if cfg.require_distinct_nvidia_keys:
        fingerprints = [value for _, value in nvidia_fingerprints]
        if len(fingerprints) != len(set(fingerprints)):
            raise RuntimeError(
                "The five configured NVIDIA model lanes must resolve to five distinct authorized key values."
            )
    if probe_endpoints:
        # Distinct model/key lanes are independent, so capability probes run concurrently.
        with ThreadPoolExecutor(max_workers=len(specs)) as pool:
            futures = {pool.submit(probe_model_endpoint, spec, cfg): spec.name for spec in specs}
            probe_results = {futures[future]: future.result() for future in as_completed(futures)}
        for row in rows:
            row.update(probe_results[row["model"]])
        frame = pd.DataFrame(rows)
        display(frame)
        failed = frame[frame["endpoint"] != "reachable"]
        if not failed.empty:
            details = failed[["model", "endpoint", "probe_error"]].to_dict("records")
            raise RuntimeError(
                "Model endpoint preflight failed. No benchmark trajectories were started. "
                f"Fix or replace these endpoints: {details}"
            )
    else:
        display(frame)
    return frame


## 6. ReAct-style controller and trajectory collection


In [ ]:
REACT_SYSTEM_PROMPT = """You are a mathematical problem-solving agent with one Python tool.

At each turn, return exactly one action and no surrounding discussion:
- A line beginning with `Reasoning step:` followed by one fenced `python` block containing
  executable Python that prints the required result; or
- A line beginning with `Reasoning step:` followed by one fenced `answer` block containing
  only a concrete final mathematical answer.

Rules:
- Never invent, predict, or write an Observation. The controller supplies real execution output.
- Never copy format descriptions such as `Python code`, `final answer only`, or template text.
- Use only math libraries available in the sandbox: math, fractions, decimal, itertools,
  collections, sympy, numpy, and scipy.
- Each code action must make new progress or perform a useful verification.
- Do not repeat a failed action unchanged.
- Use no more than four Python actions. Once enough evidence exists, answer immediately.
- The answer block must contain the answer to the original problem, not an intermediate value.
- Diagrams are not necessarily to scale. Asymptote/TikZ coordinates are rendering instructions,
  not mathematical givens; use only stated labels, relations, and problem conditions.
"""


@dataclass
class StepRecord:
    step_num: int
    model_input_messages: List[Dict] = field(default_factory=list)
    model_input_hash: str = ""
    reasoning: str = ""
    provider_reasoning: str = ""
    raw_response: str = ""
    action_type: str = "none"
    action_source: str = ""
    parse_error: str = ""
    code: str = ""
    final_answer: str = ""
    stdout: str = ""
    stderr: str = ""
    observation: str = ""
    code_success: bool = False
    code_blocked: bool = False
    tool_elapsed_sec: float = 0.0
    request_latency_sec: float = 0.0
    service_latency_sec: float = 0.0
    service_time_all_attempts_sec: float = 0.0
    rate_limit_wait_sec: float = 0.0
    retry_wait_sec: float = 0.0
    request_attempts: int = 0
    retry_errors: List[Dict] = field(default_factory=list)
    finish_reason: str = ""
    provider_truncated: bool = False
    visible_content_chars: int = 0
    provider_reasoning_chars: int = 0
    input_tokens: int = 0
    output_tokens: int = 0
    hidden_reasoning_tokens: int = 0
    evidence_hash: str = ""
    error: str = ""


@dataclass
class Trajectory:
    problem_id: str
    dataset: str
    subject: str
    level: str
    problem_text: str
    gold_answer: str
    agent_name: str
    provider: str
    model_id: str
    steps: List[Dict] = field(default_factory=list)
    final_answer: str = ""
    finished: bool = False
    stop_reason: str = ""
    started_at: str = ""
    elapsed_sec: float = 0.0
    timed_out: bool = False

    @property
    def total_steps(self):
        return len(self.steps)

    def to_dict(self):
        return asdict(self)

    @classmethod
    def from_dict(cls, value: Dict):
        allowed = set(cls.__dataclass_fields__)
        return cls(**{k: v for k, v in value.items() if k in allowed})


CODE_RE = re.compile(r"```python\s*(.*?)```", re.IGNORECASE | re.DOTALL)
ANSWER_RE = re.compile(r"```answer\s*(.*?)```", re.IGNORECASE | re.DOTALL)
OPEN_CODE_RE = re.compile(r"```python\s*(.*)$", re.IGNORECASE | re.DOTALL)
REASON_RE = re.compile(r"Reasoning\s*step\s*:\s*(.*?)(?=```|$)", re.IGNORECASE | re.DOTALL)
OBS_RE = re.compile(r"(?:^|\n)\s*Observation\s*:", re.IGNORECASE)
PLACEHOLDER_MARKERS = (
    "<python code", "<final answer", "<one concise sentence",
    "python code; always print", "final answer only", "your final mathematical value",
    "template text", "final_value",
)


def sanitize_action_content(action_type: str, content: str) -> str:
    content = str(content or "").strip().strip("`").strip()
    content = re.sub(r"^(?:\\n)+|(?:\\n)+$", "", content).strip()
    if action_type == "answer":
        content = re.sub(
            r"^\s*(?:final\s+answer|answer)\s*(?:is|:|=)?\s*(?:\r?\n)+",
            "",
            content,
            flags=re.IGNORECASE,
        ).strip()
        content = re.sub(
            r"^\s*(?:final\s+answer|answer)\s*(?:is|:|=)\s*",
            "",
            content,
            flags=re.IGNORECASE,
        ).strip()
    return content


def placeholder_reason(action_type: str, content: str) -> str:
    normalized = re.sub(r"\s+", " ", str(content or "").replace("\\n", " ")).strip().lower()
    if not normalized:
        return "empty_action"
    if any(marker in normalized for marker in PLACEHOLDER_MARKERS):
        return "template_placeholder"
    if action_type == "code" and normalized in {"...", "pass", "print(...)"}:
        return "non_executable_template"
    if action_type == "code" and re.match(
        r"^(?:reasoning\s+step|answer|final\s+answer)\s*:?(?:\s|$)",
        normalized,
    ):
        return "answer_text_in_code_block"
    return ""


def concise_answer_fallback(text: str) -> str:
    text = str(text or "").strip()
    bold_labelled = list(re.finditer(
        r"(?:^|\n)\s*\*{1,2}(?:final\s+)?answer\s*:\*{1,2}"
        r"\s*(?:\r?\n)+\s*([^\n]{1,100})",
        text,
        flags=re.IGNORECASE,
    ))
    if bold_labelled:
        return sanitize_action_content("answer", bold_labelled[-1].group(1))
    labelled_matches = list(re.finditer(
        r"(?:final\s+answer|answer)\s*(?:is|:|=)\s*([^\n]{1,100})",
        text,
        flags=re.IGNORECASE,
    ))
    if labelled_matches:
        return sanitize_action_content("answer", labelled_matches[-1].group(1))
    boxed = re.search(r"\\boxed\{([^{}]+)\}", text)
    if boxed:
        return sanitize_action_content("answer", boxed.group(1))
    markdown_answers = list(re.finditer(
        r"(?:^|\n)\s*\*{0,2}(?:final\s+)?answer\*{0,2}\s*:?[ \t]*\n+(.+)$",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    ))
    if markdown_answers:
        candidate = markdown_answers[-1].group(1).strip()
        candidate = re.sub(r"^(?:\\\[|\$\$?)\s*", "", candidate)
        candidate = re.sub(r"\s*(?:\\\]|\$\$?)$", "", candidate).strip()
        if (
            0 < len(candidate) <= 200
            and len(candidate.splitlines()) <= 4
            and not any(token in candidate.lower() for token in ("```", "import ", "print("))
        ):
            return sanitize_action_content("answer", candidate)
    generic_fences = list(re.finditer(
        r"```(?:text|latex|math)?\s*\n?\s*([^`]{1,100}?)\s*```",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    ))
    if generic_fences:
        candidate = generic_fences[-1].group(1).strip()
        if (
            len(candidate.splitlines()) <= 2
            and not any(token in candidate.lower() for token in ("import ", "print(", "def ", "="))
        ):
            return sanitize_action_content("answer", candidate)
    displays = list(re.finditer(r"\\\[(.{1,300}?)\\\]", text, flags=re.DOTALL))
    if displays:
        candidate = re.sub(r"\s+", " ", displays[-1].group(1)).strip()
        if "=" in candidate:
            candidate = candidate.rsplit("=", 1)[-1].strip()
        if 0 < len(candidate) <= 100 and not any(
            token in candidate.lower() for token in ("import ", "print(", "def ")
        ):
            return sanitize_action_content("answer", candidate)
    if len(text) <= 80 and len(text.splitlines()) <= 2 and re.search(r"\d", text):
        if not any(token in text.lower() for token in ("reasoning", "python", "import ", "print(")):
            return sanitize_action_content("answer", text)
    return ""


def parse_action(raw: str, provider_reasoning: str = "") -> Dict:
    """Parse only visible assistant content; provider reasoning remains audit metadata."""
    raw = str(raw or "")
    marker = OBS_RE.search(raw)
    clean = raw[:marker.start()] if marker else raw
    reasoning_match = REASON_RE.search(clean)
    reasoning = reasoning_match.group(1).strip() if reasoning_match else ""
    code_match = CODE_RE.search(clean)
    answer_match = ANSWER_RE.search(clean)
    candidates = []
    if code_match:
        candidates.append((code_match.start(), "code", code_match.group(1).strip()))
    if answer_match:
        candidates.append((answer_match.start(), "answer", answer_match.group(1).strip()))
    if not candidates:
        open_code = OPEN_CODE_RE.search(clean)
        if open_code and open_code.group(1).strip():
            candidates.append((open_code.start(), "code", open_code.group(1).strip().rstrip("`")))
    if len({kind for _, kind, _ in candidates}) > 1:
        return {
            "reasoning": reasoning,
            "action_type": "none",
            "content": "",
            "action_source": "visible_content",
            "parse_error": "multiple_actions",
        }
    if not candidates:
        fallback_answer = concise_answer_fallback(clean)
        placeholder = placeholder_reason("answer", fallback_answer) if fallback_answer else ""
        if fallback_answer and not placeholder:
            return {
                "reasoning": reasoning,
                "action_type": "answer",
                "content": fallback_answer,
                "action_source": "visible_fallback",
                "parse_error": "",
            }
        return {
            "reasoning": reasoning,
            "action_type": "none",
            "content": "",
            "action_source": "",
            "parse_error": placeholder or "no_visible_action",
        }
    _, action_type, content = sorted(candidates, key=lambda x: x[0])[0]
    content = sanitize_action_content(action_type, content)
    placeholder = placeholder_reason(action_type, content)
    if placeholder:
        return {
            "reasoning": reasoning,
            "action_type": "none",
            "content": "",
            "action_source": "visible_content",
            "parse_error": placeholder,
        }
    return {
        "reasoning": reasoning,
        "action_type": action_type,
        "content": content,
        "action_source": "visible_content",
        "parse_error": "",
    }


def assistant_turn(step: Dict) -> str:
    if step["action_type"] == "code":
        return f"Reasoning step: {step['reasoning']}\n```python\n{step['code']}\n```"
    if step["action_type"] == "answer":
        return f"Reasoning step: {step['reasoning']}\n```answer\n{step['final_answer']}\n```"
    return step.get("raw_response", "")


def build_messages(
    problem: str,
    steps: List[Dict],
    spec: ModelSpec,
    cfg: Config,
    force_final_answer: bool = False,
    prefer_final_answer: bool = False,
) -> List[Dict]:
    system_content = REACT_SYSTEM_PROMPT
    if spec.system_directive:
        system_content = spec.system_directive.strip() + "\n\n" + system_content
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": f"Solve this problem:\n\n{problem}"},
    ]
    for step in steps:
        messages.append({"role": "assistant", "content": assistant_turn(step)})
        if step["action_type"] == "code":
            messages.append({"role": "user", "content": "Observation:\n" + step["observation"]})
        elif step["action_type"] == "none":
            feedback = step.get("observation") or step.get("error") or "No valid action was found."
            messages.append({
                "role": "user",
                "content": (
                    "Controller feedback: " + feedback + "\n"
                    "Return one concrete fenced Python action or one concrete fenced answer. "
                    "Do not repeat template wording or the previous action."
                ),
            })
    if force_final_answer:
        messages.append({
            "role": "user",
            "content": (
                "Tool-action budget reached. Do not emit more Python. Using the problem and "
                "real observations already present, emit exactly one concise fenced ```answer``` "
                "block containing the final answer."
            ),
        })
    elif prefer_final_answer:
        messages.append({
            "role": "user",
            "content": (
                "A real tool result is now available. If it resolves the original problem, "
                "return the concrete final answer now; otherwise make exactly one new useful action."
            ),
        })
    # Bound context while keeping the system prompt and problem. Full trace remains on disk.
    while sum(len(x["content"]) for x in messages) > cfg.max_history_chars and len(messages) > 4:
        del messages[2:4]
    return messages


_print_lock = threading.Lock()


def live_event(problem_id: str, model: str, step: int, title: str, body: str = ""):
    if not cfg.live_trace:
        return
    with _print_lock:
        tqdm.write(f"[{problem_id}] [{model}] step {step} | {title}")
        if body:
            tqdm.write(truncate_middle(body, 1800))


def run_react_trajectory(problem: Dict, spec: ModelSpec, cfg: Config) -> Trajectory:
    start = time.perf_counter()
    trajectory_deadline = time.monotonic() + cfg.trajectory_timeout_sec
    traj = Trajectory(
        problem_id=problem["id"],
        dataset=problem["dataset"],
        subject=problem["subject"],
        level=problem["level"],
        problem_text=problem["problem"],
        gold_answer=problem["gold_answer"],
        agent_name=spec.name,
        provider=spec.provider,
        model_id=spec.model_id,
        started_at=datetime.now(timezone.utc).isoformat(),
    )
    try:
        with MultiStepPythonSandbox(cfg) as sandbox:
            executed_tool_actions = 0
            consecutive_format_errors = 0
            consecutive_code_failures = 0
            repeated_actions = 0
            answer_only_violations = 0
            previous_action_signature = ""
            force_final_next = False
            prefer_final_next = False
            for step_num in range(1, cfg.max_steps + 1):
                if time.monotonic() >= trajectory_deadline:
                    traj.stop_reason = "trajectory_timeout"
                    traj.timed_out = True
                    live_event(problem["id"], spec.name, step_num, "TRAJECTORY TIMEOUT")
                    break
                force_final_answer = (
                    force_final_next
                    or
                    executed_tool_actions >= cfg.max_tool_actions
                    or step_num == cfg.max_steps
                )
                messages = build_messages(
                    problem["problem"],
                    traj.steps,
                    spec,
                    cfg,
                    force_final_answer=force_final_answer,
                    prefer_final_answer=prefer_final_next and not force_final_answer,
                )
                live_event(
                    problem["id"],
                    spec.name,
                    step_num,
                    "API REQUEST",
                    f"deadline={cfg.request_total_timeout_sec}s; trajectory remaining="
                    f"{max(0.0, trajectory_deadline - time.monotonic()):.1f}s",
                )
                try:
                    raw, usage = call_model(
                        spec,
                        messages,
                        cfg,
                        deadline_monotonic=trajectory_deadline,
                        context=f"{problem['id']} | {spec.name} | step {step_num}",
                    )
                except Exception as exc:
                    if time.monotonic() >= trajectory_deadline:
                        traj.stop_reason = "trajectory_timeout"
                        traj.timed_out = True
                    else:
                        traj.stop_reason = "api_error"
                    failure = getattr(exc, "meta", {})
                    traj.steps.append(asdict(StepRecord(
                        step_num=step_num,
                        error=str(exc),
                        request_latency_sec=float(failure.get("latency_sec", 0.0)),
                        service_latency_sec=float(failure.get("service_latency_sec", 0.0)),
                        service_time_all_attempts_sec=float(
                            failure.get("service_time_all_attempts_sec", 0.0)
                        ),
                        rate_limit_wait_sec=float(failure.get("rate_limit_wait_sec", 0.0)),
                        retry_wait_sec=float(failure.get("retry_wait_sec", 0.0)),
                        request_attempts=int(failure.get("request_attempts", 0)),
                        retry_errors=list(failure.get("retry_errors", [])),
                        finish_reason=str(failure.get("finish_reason", "error")),
                    )))
                    live_event(problem["id"], spec.name, step_num, "API ERROR", str(exc))
                    break

                action = parse_action(raw, usage.get("provider_reasoning", ""))
                serialized_input = json.dumps(messages, ensure_ascii=True, sort_keys=True)
                record = StepRecord(
                    step_num=step_num,
                    model_input_messages=messages if cfg.store_full_model_inputs else [],
                    model_input_hash=hashlib.sha256(serialized_input.encode()).hexdigest()[:16],
                    reasoning=action["reasoning"],
                    provider_reasoning=usage.get("provider_reasoning", ""),
                    raw_response=raw,
                    action_type=action["action_type"],
                    action_source=action.get("action_source", ""),
                    parse_error=action.get("parse_error", ""),
                    request_latency_sec=float(usage.get("latency_sec", 0.0)),
                    service_latency_sec=float(usage.get("service_latency_sec", 0.0)),
                    service_time_all_attempts_sec=float(
                        usage.get("service_time_all_attempts_sec", 0.0)
                    ),
                    rate_limit_wait_sec=float(usage.get("rate_limit_wait_sec", 0.0)),
                    retry_wait_sec=float(usage.get("retry_wait_sec", 0.0)),
                    request_attempts=int(usage.get("request_attempts", 1)),
                    retry_errors=list(usage.get("retry_errors", [])),
                    finish_reason=str(usage.get("finish_reason", "")),
                    provider_truncated=bool(usage.get("provider_truncated", False)),
                    visible_content_chars=len(raw),
                    provider_reasoning_chars=len(usage.get("provider_reasoning", "")),
                    input_tokens=int(usage.get("input_tokens", 0)),
                    output_tokens=int(usage.get("output_tokens", 0)),
                    hidden_reasoning_tokens=int(usage.get("hidden_reasoning_tokens", 0)),
                )
                live_event(
                    problem["id"],
                    spec.name,
                    step_num,
                    "API RESPONSE",
                    (
                        f"total={record.request_latency_sec:.2f}s, "
                        f"service={record.service_time_all_attempts_sec:.2f}s, "
                        f"rpm_wait={record.rate_limit_wait_sec:.2f}s, "
                        f"retry_wait={record.retry_wait_sec:.2f}s, "
                        f"attempts={record.request_attempts}, finish={record.finish_reason or 'unknown'}"
                    ),
                )
                trace_reasoning = record.reasoning
                if record.provider_reasoning:
                    trace_reasoning += "\nProvider reasoning channel:\n" + record.provider_reasoning
                live_event(problem["id"], spec.name, step_num, "REASONING", trace_reasoning)

                if action["action_type"] == "answer":
                    consecutive_format_errors = 0
                    record.final_answer = action["content"]
                    traj.final_answer = action["content"]
                    traj.finished = bool(traj.final_answer.strip())
                    traj.stop_reason = "explicit_final_answer"
                    traj.steps.append(asdict(record))
                    live_event(problem["id"], spec.name, step_num, "FINAL ANSWER", traj.final_answer)
                    break

                if action["action_type"] == "code":
                    record.code = action["content"]
                    if force_final_answer:
                        answer_only_violations += 1
                        record.action_type = "none"
                        record.error = "Final-answer-only instruction was ignored"
                        record.observation = (
                            "No code executed: a concrete final answer was required on this turn."
                        )
                        traj.steps.append(asdict(record))
                        live_event(problem["id"], spec.name, step_num, "ANSWER REQUIRED", record.code)
                        force_final_next = True
                        prefer_final_next = False
                        if answer_only_violations >= cfg.max_answer_only_violations:
                            traj.stop_reason = "answer_only_violation_limit"
                            break
                        continue
                    action_signature = hashlib.sha256(
                        re.sub(r"\s+", "", record.code).encode()
                    ).hexdigest()
                    if action_signature == previous_action_signature:
                        repeated_actions += 1
                        record.action_type = "none"
                        record.error = "Repeated code action was not executed"
                        record.observation = "No code executed: exact repeated action made no new progress."
                        traj.steps.append(asdict(record))
                        live_event(problem["id"], spec.name, step_num, "REPEATED ACTION", record.code)
                        force_final_next = True
                        prefer_final_next = False
                        if repeated_actions >= cfg.max_repeated_actions:
                            traj.stop_reason = "repeated_action_limit"
                            break
                        continue
                    previous_action_signature = action_signature
                    repeated_actions = 0
                    consecutive_format_errors = 0
                    force_final_next = False
                    prefer_final_next = False
                    live_event(problem["id"], spec.name, step_num, "CODE", record.code)
                    result = sandbox.run(record.code)
                    executed_tool_actions += 1
                    record.stdout = result.stdout
                    record.stderr = result.stderr
                    record.code_success = result.ok
                    record.code_blocked = result.blocked
                    record.tool_elapsed_sec = result.elapsed_sec
                    record.observation = result.observation(cfg.max_observation_chars)
                    record.evidence_hash = hashlib.sha256(
                        (record.code + "\n" + record.observation).encode()
                    ).hexdigest()[:16]
                    traj.steps.append(asdict(record))
                    live_event(problem["id"], spec.name, step_num, "OBSERVATION", record.observation)
                    if result.ok:
                        consecutive_code_failures = 0
                        prefer_final_next = bool(result.stdout.strip())
                    else:
                        consecutive_code_failures += 1
                        if consecutive_code_failures >= 2:
                            force_final_next = True
                    continue

                record.error = "No valid action parsed: " + (record.parse_error or "unknown")
                if record.provider_truncated:
                    record.observation = (
                        "No action accepted: the provider stopped at its output limit before "
                        "returning a concrete visible action."
                    )
                elif record.parse_error == "template_placeholder":
                    record.observation = (
                        "No action accepted: template placeholder text is neither executable "
                        "code nor a concrete final answer."
                    )
                else:
                    record.observation = (
                        "No action accepted: response did not contain one concrete visible action "
                        f"({record.parse_error or 'unknown parse error'})."
                    )
                traj.steps.append(asdict(record))
                format_body = raw or record.provider_reasoning
                live_event(problem["id"], spec.name, step_num, "FORMAT ERROR", format_body)
                consecutive_format_errors += 1
                force_final_next = True
                prefer_final_next = False
                if consecutive_format_errors >= cfg.max_consecutive_format_errors:
                    traj.stop_reason = "format_error_limit"
                    break
            else:
                traj.stop_reason = "max_steps"
    finally:
        traj.elapsed_sec = time.perf_counter() - start
    # Never promote an intermediate tool value to final_answer. Only an explicit answer action counts.
    return traj


In [ ]:
# Deterministic controller regressions from observed provider outputs.
placeholder_code = parse_action(
    "Reasoning step: compute\n```python\n<Python code; always print the result you need>\n```"
)
placeholder_answer = parse_action(
    "Reasoning step: conclude\n```answer\n\\n<final answer only>\\n\n```"
)
prefixed_answer = parse_action("Reasoning step: conclude\n```answer\nanswer\n48\n```")
provider_only = parse_action("", "Thus the final answer is 10.")
multiple_actions = parse_action(
    "Reasoning step: compute\n```python\nprint(1)\n```\n```answer\n1\n```"
)
mislabeled_reasoning = parse_action("```python\nReasoning step: use the area formula\n```")
mislabeled_answer = parse_action("```python\nanswer\nmu = 1/(2*n + 1)\n```")
markdown_answer = parse_action("**Answer**\n\n\\[p-q\\]")
bold_colon_answer = parse_action("**Reasoning step:** done\n\n**Answer:**  \n1")
generic_fenced_answer = parse_action("```\n3\n```")
plain_display_answer = parse_action(
    "The double series equals the following.\n\\[\\sum_{j,k}a_{j,k}=p-q\\]"
)

assert placeholder_code["parse_error"] == "template_placeholder"
assert placeholder_answer["parse_error"] == "template_placeholder"
assert prefixed_answer["action_type"] == "answer" and prefixed_answer["content"] == "48"
assert provider_only["action_type"] == "none"  # Reasoning metadata is not a final answer span.
assert multiple_actions["parse_error"] == "multiple_actions"
assert mislabeled_reasoning["parse_error"] == "answer_text_in_code_block"
assert mislabeled_answer["parse_error"] == "answer_text_in_code_block"
assert markdown_answer["action_type"] == "answer" and markdown_answer["content"] == "p-q"
assert bold_colon_answer["action_type"] == "answer" and bold_colon_answer["content"] == "1"
assert generic_fenced_answer["action_type"] == "answer" and generic_fenced_answer["content"] == "3"
assert plain_display_answer["action_type"] == "answer" and plain_display_answer["content"] == "p-q"
print("Controller self-test passed: placeholders and mislabeled code rejected; visible-answer policy enforced.")


In [ ]:
class JSONLCheckpoint:
    def __init__(self, output_dir: str, run_name: str, specs: List[ModelSpec]):
        self.root = Path(output_dir) / "checkpoints" / run_name
        self.root.mkdir(parents=True, exist_ok=True)
        self.paths = {s.name: self.root / f"{s.name}.jsonl" for s in specs}
        self.lock = threading.Lock()

    def load(self) -> Dict[str, Dict[str, Trajectory]]:
        data = {name: {} for name in self.paths}
        for name, path in self.paths.items():
            if not path.exists():
                continue
            for line_number, line in enumerate(
                path.read_text(encoding="utf-8").splitlines(),
                start=1,
            ):
                if line.strip():
                    try:
                        traj = Trajectory.from_dict(json.loads(line))
                    except Exception as exc:
                        tqdm.write(
                            f"Skipping corrupt checkpoint line {path.name}:{line_number}: {exc}"
                        )
                        continue
                    data[name][traj.problem_id] = traj
        return data

    def append(self, traj: Trajectory):
        path = self.paths[traj.agent_name]
        with self.lock, path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(traj.to_dict(), ensure_ascii=True) + "\n")
            handle.flush()
            os.fsync(handle.fileno())

    def clear(self):
        for path in self.paths.values():
            if path.exists():
                path.unlink()


def retryable_infrastructure_checkpoint(traj: Trajectory) -> bool:
    if traj.stop_reason in {"worker_exception", "trajectory_timeout"}:
        return True
    if traj.stop_reason != "api_error":
        return False
    errors = [
        error
        for step in traj.steps
        for error in (step.get("retry_errors") or [])
    ]
    if not errors:
        return True
    retryable_statuses = {408, 409, 429, 500, 502, 503, 504}
    retryable_types = {"APIConnectionError", "APITimeoutError", "RateLimitError", "InternalServerError"}
    return any(
        error.get("status") in retryable_statuses or error.get("type") in retryable_types
        for error in errors
    )


def run_experiment(
    problems: List[Dict],
    specs: List[ModelSpec],
    cfg: Config,
    run_name: str,
    live: bool = True,
    resume: bool = True,
) -> Dict[str, List[Trajectory]]:
    """Run one independent sequential problem lane per model, with all model lanes concurrent."""
    old_live = cfg.live_trace
    cfg.live_trace = live
    checkpoint = JSONLCheckpoint(cfg.output_dir, run_name, specs)
    if not resume:
        checkpoint.clear()
    existing = checkpoint.load() if resume else {s.name: {} for s in specs}
    retrying = []
    if resume and cfg.retry_infrastructure_failures_on_resume:
        for spec in specs:
            for problem_id, traj in list(existing[spec.name].items()):
                if retryable_infrastructure_checkpoint(traj):
                    retrying.append((spec.name, problem_id, traj.stop_reason))
                    del existing[spec.name][problem_id]
        if retrying:
            print(
                f"Retrying {len(retrying)} infrastructure-failed checkpoint trajectories; "
                f"first entries: {retrying[:10]}"
            )
    problem_ids = {problem["id"] for problem in problems}
    already_complete = sum(
        problem_id in existing[spec.name]
        for spec in specs
        for problem_id in problem_ids
    )
    progress = tqdm(
        total=len(problems) * len(specs),
        initial=already_complete,
        desc=f"{run_name}: generated trajectories",
        unit="traj",
        dynamic_ncols=True,
    )

    def run_model_lane(spec: ModelSpec):
        """One key/model processes its problems sequentially; different model lanes overlap."""
        consecutive_infrastructure_failures = 0
        for problem_index, problem in enumerate(problems, start=1):
            if problem["id"] in existing[spec.name]:
                continue
            tqdm.write(
                f"[{spec.name}] starting problem {problem_index}/{len(problems)}: {problem['id']}"
            )
            try:
                traj = run_react_trajectory(problem, spec, cfg)
            except Exception:
                traj = Trajectory(
                    problem_id=problem["id"], dataset=problem["dataset"],
                    subject=problem["subject"], level=problem["level"],
                    problem_text=problem["problem"], gold_answer=problem["gold_answer"],
                    agent_name=spec.name, provider=spec.provider, model_id=spec.model_id,
                    stop_reason="worker_exception",
                    steps=[asdict(StepRecord(step_num=0, error=traceback.format_exc()))],
                )
            checkpoint.append(traj)
            existing[spec.name][problem["id"]] = traj
            with _print_lock:
                progress.update(1)
                progress.set_postfix(
                    lane=spec.name,
                    problem=f"{problem_index}/{len(problems)}",
                    steps=traj.total_steps,
                    sec=f"{traj.elapsed_sec:.1f}",
                    finished=traj.finished,
                    refresh=True,
                )
            tqdm.write(
                f"[{spec.name}] checkpointed problem {problem_index}/{len(problems)}: "
                f"steps={traj.total_steps}, finished={traj.finished}, stop={traj.stop_reason}"
            )
            if retryable_infrastructure_checkpoint(traj):
                consecutive_infrastructure_failures += 1
                cooldown = (
                    cfg.lane_breaker_cooldown_sec
                    if consecutive_infrastructure_failures >= cfg.lane_breaker_after_failures
                    else cfg.lane_api_error_cooldown_sec
                )
                tqdm.write(
                    f"[{spec.name}] infrastructure cooldown {cooldown:.0f}s after "
                    f"{consecutive_infrastructure_failures} consecutive failure(s)"
                )
                time.sleep(cooldown)
            else:
                consecutive_infrastructure_failures = 0

    try:
        with ThreadPoolExecutor(max_workers=min(cfg.parallel_models, len(specs))) as pool:
            futures = {pool.submit(run_model_lane, spec): spec for spec in specs}
            for future in as_completed(futures):
                spec = futures[future]
                future.result()
                tqdm.write(f"[{spec.name}] model lane complete")
    finally:
        progress.close()
        cfg.live_trace = old_live
    ordered = {}
    order = {p["id"]: i for i, p in enumerate(problems)}
    for spec in specs:
        selected = [
            traj for problem_id, traj in existing[spec.name].items()
            if problem_id in order
        ]
        ordered[spec.name] = sorted(selected, key=lambda t: order[t.problem_id])
    return ordered


## 7. Recovery input and strict selection policy


In [ ]:
SOURCE_GENERATION_INPUT = (
    "/path/to/your/project/"
    "i-got-these-reviews-from-a/strive_canonical_merge/"
    "paper_math200_olympiad100_v11_canonical_generation_20260714T132605Z.zip"
)
from IPython.display import FileLink, display


def show_download_link(path):
    display(FileLink(str(Path(path).resolve())))


RECOVERY_RUN_NAME = "paper_math200_olympiad100_v11_all_invalid_recovery"

# Every invalid source record receives at most this many new trajectory attempts.
# Rerunning the recovery cell resumes from the append-only checkpoint.
MAX_RECOVERY_ATTEMPTS_PER_TRAJECTORY = 3
RERUN_PROTOCOL_FAILURES = True
SHOW_LIVE_TRACE = False
RUN_ENDPOINT_PREFLIGHT = False
REQUIRE_ALL_PUBLISHABLE = True

INFRASTRUCTURE_STOPS = {"api_error", "worker_exception", "trajectory_timeout"}
PROTOCOL_STOPS = {"format_error_limit", "answer_only_violation_limit"}
NVIDIA_RECOVERY_KEY_SLOTS = [f"NVIDIA_API_KEY_{index}" for index in range(1, 7)]

recovery_cfg = replace(cfg)
recovery_cfg.output_dir = str(Path(
    "/kaggle/working/strive_all_invalid_recovery"
    if Path("/kaggle/working").exists()
    else Path.cwd() / "strive_all_invalid_recovery"
).resolve())
Path(recovery_cfg.output_dir).mkdir(parents=True, exist_ok=True)
Path(recovery_cfg.output_dir, "checkpoints").mkdir(parents=True, exist_ok=True)

# The router owns NVIDIA retries and rotates six distinct accounts.
recovery_cfg.max_retries = 1
recovery_cfg.request_timeout_sec = 180
recovery_cfg.request_total_timeout_sec = 1200
recovery_cfg.trajectory_timeout_sec = 1800
recovery_cfg.rate_limit_cooldown_sec = 180.0
recovery_cfg.resource_exhausted_cooldown_sec = 60.0
recovery_cfg.rpm_utilization = 0.80
recovery_cfg.live_trace = SHOW_LIVE_TRACE
cfg.live_trace = SHOW_LIVE_TRACE

print("Recovery output directory:", recovery_cfg.output_dir)


In [ ]:
def _safe_extract_generation_zip(path, target):
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    safe_root = target.resolve()
    import zipfile
    with zipfile.ZipFile(path) as archive:
        for member in archive.infolist():
            destination = (target / member.filename).resolve()
            if destination != safe_root and safe_root not in destination.parents:
                raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
        archive.extractall(target)


def load_generation_source(value):
    source = Path(value).expanduser().resolve()
    if not source.exists():
        raise FileNotFoundError(source)
    if source.is_file():
        if source.suffix.lower() != ".zip":
            raise ValueError("SOURCE_GENERATION_INPUT must be a ZIP or directory")
        target = Path(recovery_cfg.output_dir) / "loaded_source" / source.stem
        _safe_extract_generation_zip(source, target)
        source = target
    manifests = list(source.rglob("generation_manifest.json"))
    if len(manifests) != 1:
        raise RuntimeError(
            f"Expected one generation_manifest.json under {source}; found {len(manifests)}"
        )
    root = manifests[0].parent
    manifest = json.loads(manifests[0].read_text(encoding="utf-8"))
    for name, expected in manifest.get("sha256", {}).items():
        candidate = root / name
        if not candidate.exists():
            raise FileNotFoundError(candidate)
        actual = hashlib.sha256(candidate.read_bytes()).hexdigest()
        if actual != expected:
            raise RuntimeError(f"Checksum mismatch: {candidate}")
    with (root / "trajectories.jsonl").open(encoding="utf-8") as handle:
        trajectories = [
            Trajectory.from_dict(json.loads(line)) for line in handle if line.strip()
        ]
    with (root / "problems.jsonl").open(encoding="utf-8") as handle:
        problems = [json.loads(line) for line in handle if line.strip()]
    return trajectories, problems, manifest


def trajectory_validation_reasons(trajectory):
    reasons = []
    if not trajectory.finished:
        reasons.append("not_finished")
    if trajectory.timed_out:
        reasons.append("timed_out")
    if trajectory.stop_reason != "explicit_final_answer":
        reasons.append(trajectory.stop_reason or "missing_stop_reason")
    if not str(trajectory.final_answer or "").strip():
        reasons.append("empty_final_answer")
    if not trajectory.steps:
        reasons.append("no_steps")
    if not any(
        str(step.get("action_type", "")) == "answer"
        and str(step.get("final_answer", step.get("action_content", "")) or "").strip()
        for step in trajectory.steps
    ):
        reasons.append("answer_step_missing")
    return sorted(set(reasons))


def is_publishable_trajectory(trajectory):
    return not trajectory_validation_reasons(trajectory)


source_trajectories, source_problems, source_manifest = load_generation_source(
    SOURCE_GENERATION_INPUT
)
source_by_key = {}
for trajectory in source_trajectories:
    key = (trajectory.agent_name, trajectory.problem_id)
    if key in source_by_key:
        raise RuntimeError(f"Duplicate source trajectory: {key}")
    source_by_key[key] = trajectory
problems_by_id = {str(problem["id"]): problem for problem in source_problems}

if len(source_by_key) != 1800:
    raise RuntimeError(f"Expected 1,800 source trajectories, found {len(source_by_key)}")
if len(problems_by_id) != 300:
    raise RuntimeError(f"Expected 300 source problems, found {len(problems_by_id)}")

required_reruns = []
for key, trajectory in source_by_key.items():
    reasons = trajectory_validation_reasons(trajectory)
    if not reasons:
        continue
    if not RERUN_PROTOCOL_FAILURES and trajectory.stop_reason in PROTOCOL_STOPS:
        continue
    required_reruns.append({
        "agent_name": key[0],
        "problem_id": key[1],
        "dataset": trajectory.dataset,
        "subject": trajectory.subject,
        "level": trajectory.level,
        "original_stop_reason": trajectory.stop_reason,
        "validation_reasons": "|".join(reasons),
    })
required_reruns.sort(key=lambda item: (item["agent_name"], item["problem_id"]))
required_reruns_df = pd.DataFrame(required_reruns)
display(required_reruns_df.groupby(
    ["agent_name", "original_stop_reason"]
).size().rename("count").reset_index())
print("Total trajectories selected for recovery:", len(required_reruns))

selection_path = Path(recovery_cfg.output_dir) / "required_reruns.csv"
required_reruns_df.to_csv(selection_path, index=False)
print("Selection audit:", selection_path)


## 8. Six-key NVIDIA router and resumable recovery engine


In [ ]:
missing_keys = [
    name for name in NVIDIA_RECOVERY_KEY_SLOTS
    if not os.environ.get(name, "").strip()
]
if missing_keys:
    raise RuntimeError(f"Configure all six NVIDIA recovery keys: {missing_keys}")
fingerprints = [
    hashlib.sha256(os.environ[name].strip().encode()).hexdigest()
    for name in NVIDIA_RECOVERY_KEY_SLOTS
]
if len(set(fingerprints)) != len(fingerprints):
    raise RuntimeError("The six NVIDIA recovery credentials must be distinct")
if any(item["agent_name"] == "gpt-5-nano" for item in required_reruns):
    if not os.environ.get("OPENAI_API_KEY", "").strip():
        raise RuntimeError("OPENAI_API_KEY is required for GPT-5 nano recovery")
print("Recovery credentials validated; secret values remain hidden.")

PER_KEY_RPM = 2
NVIDIA_MODEL_RPM = {
    "ministral-14b": 3,
    "glm-5.2": 3,
    "minimax-m3": 3,
    "nemotron-3-nano": 6,
    "gpt-oss-20b": 3,
}
MAX_ROUTER_ATTEMPTS_PER_CALL = 12
KEY_429_COOLDOWN_SEC = 180.0
TRANSIENT_KEY_COOLDOWN_SEC = 60.0
MINIMAX_DEGRADED_COOLDOWN_SEC = 300.0
BETWEEN_RECOVERY_PASSES_SEC = 30.0


class NvidiaRecoveryRouter:
    def __init__(self, key_slots):
        self.key_slots = list(key_slots)
        self.condition = threading.Condition()
        self.busy = {slot: False for slot in self.key_slots}
        self.cooldown_until = {slot: 0.0 for slot in self.key_slots}
        self.cursor = 0
        self.model_pacers = {
            name: PacedRateLimiter(rpm, recovery_cfg.rpm_utilization)
            for name, rpm in NVIDIA_MODEL_RPM.items()
        }
        self.model_cooldown_until = defaultdict(float)

    def acquire_key(self, deadline):
        with self.condition:
            while True:
                now = time.monotonic()
                for offset in range(len(self.key_slots)):
                    index = (self.cursor + offset) % len(self.key_slots)
                    slot = self.key_slots[index]
                    if not self.busy[slot] and self.cooldown_until[slot] <= now:
                        self.busy[slot] = True
                        self.cursor = (index + 1) % len(self.key_slots)
                        return slot
                remaining = deadline - now
                if remaining <= 1:
                    raise TimeoutError("No NVIDIA recovery key available before deadline")
                available_times = [
                    value for slot, value in self.cooldown_until.items()
                    if not self.busy[slot]
                ]
                wait = min(5.0, remaining - 1)
                if available_times:
                    wait = min(wait, max(0.1, min(available_times) - now))
                self.condition.wait(timeout=max(0.1, wait))

    def release_key(self, slot, cooldown=0.0):
        with self.condition:
            self.busy[slot] = False
            self.cooldown_until[slot] = max(
                self.cooldown_until[slot], time.monotonic() + float(cooldown)
            )
            self.condition.notify_all()

    @staticmethod
    def failure_blob(exc):
        parts = [str(exc)]
        if isinstance(exc, ModelCallError):
            parts.append(json.dumps(exc.meta.get("retry_errors", []), sort_keys=True))
        return " ".join(parts)

    def wait_for_model(self, model_name, deadline):
        while self.model_cooldown_until[model_name] > time.monotonic():
            remaining = deadline - time.monotonic()
            if remaining <= 1:
                raise TimeoutError(f"{model_name} recovery deadline exhausted")
            wait = min(
                30.0,
                self.model_cooldown_until[model_name] - time.monotonic(),
                remaining - 1,
            )
            time.sleep(max(0.1, wait))

    def call(self, base_call, spec, messages, active_cfg, **kwargs):
        if spec.provider != "nvidia":
            openai_cfg = replace(active_cfg)
            openai_cfg.max_retries = 4
            return base_call(spec, messages, openai_cfg, **kwargs)

        provided_deadline = kwargs.get("deadline_monotonic")
        deadline = min(
            provided_deadline if provided_deadline is not None else float("inf"),
            time.monotonic() + recovery_cfg.request_total_timeout_sec,
        )
        errors = []
        router_wait = 0.0
        for router_attempt in range(1, MAX_ROUTER_ATTEMPTS_PER_CALL + 1):
            self.wait_for_model(spec.name, deadline)
            pace_started = time.monotonic()
            self.model_pacers[spec.name].acquire()
            router_wait += time.monotonic() - pace_started
            slot = self.acquire_key(deadline)
            routed_spec = replace(spec, api_key_env=slot, rpm=PER_KEY_RPM)
            single_cfg = replace(active_cfg)
            single_cfg.max_retries = 1
            single_cfg.request_timeout_sec = recovery_cfg.request_timeout_sec
            single_cfg.request_total_timeout_sec = recovery_cfg.request_timeout_sec
            try:
                text, meta = base_call(
                    routed_spec, messages, single_cfg, **kwargs
                )
                self.release_key(slot)
                meta = dict(meta)
                meta["router_attempts"] = router_attempt
                meta["router_key_slot"] = slot
                meta["rate_limit_wait_sec"] = float(
                    meta.get("rate_limit_wait_sec", 0.0) + router_wait
                )
                meta["retry_errors"] = errors + list(meta.get("retry_errors", []))
                return text, meta
            except Exception as exc:
                blob = self.failure_blob(exc)
                lower = blob.lower()
                is_429 = "429" in blob or "too many requests" in lower
                is_degraded = "degraded function cannot be invoked" in lower
                is_transient = any(token in lower for token in (
                    "timed out", "timeout", "500", "502", "503", "504",
                    "resourceexhausted", "internalserver", "workers busy",
                    "connection", "list index out of range", "nonetype",
                ))
                errors.append({
                    "router_attempt": router_attempt,
                    "key_slot": slot,
                    "type": type(exc).__name__,
                    "message": truncate_middle(blob, 700),
                    "rate_limit": is_429,
                    "degraded_backend": is_degraded,
                })
                if is_429:
                    self.release_key(slot, KEY_429_COOLDOWN_SEC)
                    continue
                if is_degraded and spec.name == "minimax-m3":
                    self.release_key(slot)
                    self.model_cooldown_until[spec.name] = (
                        time.monotonic() + MINIMAX_DEGRADED_COOLDOWN_SEC
                    )
                    continue
                if is_transient:
                    self.release_key(slot, TRANSIENT_KEY_COOLDOWN_SEC)
                    continue
                self.release_key(slot)
                raise

        failure_meta = {
            "latency_sec": 0.0,
            "service_latency_sec": 0.0,
            "service_time_all_attempts_sec": 0.0,
            "rate_limit_wait_sec": router_wait,
            "retry_wait_sec": 0.0,
            "request_attempts": len(errors),
            "retry_errors": errors,
            "finish_reason": "error",
            "provider_truncated": False,
            "provider_reasoning": "",
            "input_tokens": 0,
            "output_tokens": 0,
            "hidden_reasoning_tokens": 0,
        }
        raise ModelCallError(
            f"NVIDIA recovery router exhausted for {spec.name}", failure_meta
        )


class RecoveryAttemptCheckpoint:
    def __init__(self, root):
        self.path = Path(root) / "checkpoints" / RECOVERY_RUN_NAME / "attempts.jsonl"
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.lock = threading.Lock()

    def load(self):
        attempts = defaultdict(int)
        successful = {}
        if not self.path.exists():
            return attempts, successful
        for line_number, line in enumerate(
            self.path.read_text(encoding="utf-8").splitlines(), 1
        ):
            if not line.strip():
                continue
            try:
                entry = json.loads(line)
                trajectory = Trajectory.from_dict(entry["trajectory"])
            except Exception as exc:
                tqdm.write(f"Skipping checkpoint line {line_number}: {exc}")
                continue
            key = (trajectory.agent_name, trajectory.problem_id)
            attempts[key] = max(attempts[key], int(entry.get("attempt", 1)))
            if is_publishable_trajectory(trajectory):
                successful[key] = trajectory
        return attempts, successful

    def append(self, trajectory, attempt):
        entry = {
            "attempt": int(attempt),
            "saved_at": datetime.now(timezone.utc).isoformat(),
            "validation_reasons": trajectory_validation_reasons(trajectory),
            "trajectory": trajectory.to_dict(),
        }
        with self.lock, self.path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(entry, ensure_ascii=False) + "\n")
            handle.flush()
            os.fsync(handle.fileno())


recovery_router = NvidiaRecoveryRouter(NVIDIA_RECOVERY_KEY_SLOTS)
recovery_checkpoint = RecoveryAttemptCheckpoint(recovery_cfg.output_dir)
attempt_counts, successful_replacements = recovery_checkpoint.load()
required_keys = {
    (item["agent_name"], item["problem_id"]) for item in required_reruns
}
successful_replacements = {
    key: trajectory for key, trajectory in successful_replacements.items()
    if key in required_keys
}
print("Recovered from checkpoint:", len(successful_replacements), "/", len(required_keys))


In [ ]:
specs_by_name = {spec.name: spec for spec in MODEL_SPECS}
missing_specs = sorted({key[0] for key in required_keys} - set(specs_by_name))
if missing_specs:
    raise RuntimeError(f"Missing model specifications: {missing_specs}")

if RUN_ENDPOINT_PREFLIGHT:
    pending_names = sorted({key[0] for key in required_keys - set(successful_replacements)})
    try:
        display(preflight_models([specs_by_name[name] for name in pending_names]))
    except Exception as exc:
        print("Preflight warning; normal recovery retries remain active:", exc)

_base_call_model = call_model


def _recovery_call_model(spec, messages, active_cfg, **kwargs):
    return recovery_router.call(
        _base_call_model, spec, messages, active_cfg, **kwargs
    )


success_lock = threading.Lock()
total_attempts_this_run = 0
success_progress = tqdm(
    total=len(required_keys),
    initial=len(successful_replacements),
    desc="publishable recovery trajectories",
    unit="traj",
    dynamic_ncols=True,
)


def current_pending_by_model():
    grouped = defaultdict(list)
    for key in sorted(required_keys):
        if key in successful_replacements:
            continue
        if attempt_counts[key] >= MAX_RECOVERY_ATTEMPTS_PER_TRAJECTORY:
            continue
        grouped[key[0]].append(problems_by_id[key[1]])
    return grouped


def run_recovery_lane(model_name, problems):
    spec = specs_by_name[model_name]
    lane_results = []
    for index, problem in enumerate(problems, 1):
        key = (model_name, problem["id"])
        with success_lock:
            attempt_counts[key] += 1
            attempt_number = attempt_counts[key]
        tqdm.write(
            f"[{model_name}] recovery {index}/{len(problems)} | "
            f"{problem['id']} | attempt {attempt_number}/"
            f"{MAX_RECOVERY_ATTEMPTS_PER_TRAJECTORY}"
        )
        try:
            trajectory = run_react_trajectory(problem, spec, recovery_cfg)
        except Exception:
            trajectory = Trajectory(
                problem_id=problem["id"], dataset=problem["dataset"],
                subject=problem["subject"], level=problem["level"],
                problem_text=problem["problem"], gold_answer=problem["gold_answer"],
                agent_name=spec.name, provider=spec.provider, model_id=spec.model_id,
                stop_reason="worker_exception",
                steps=[asdict(StepRecord(step_num=0, error=traceback.format_exc()))],
            )
        recovery_checkpoint.append(trajectory, attempt_number)
        reasons = trajectory_validation_reasons(trajectory)
        accepted = not reasons
        if accepted:
            with success_lock:
                if key not in successful_replacements:
                    successful_replacements[key] = trajectory
                    success_progress.update(1)
        lane_results.append((key, accepted, trajectory.stop_reason, reasons))
        success_progress.set_postfix(
            model=model_name,
            stop=trajectory.stop_reason,
            accepted=accepted,
            refresh=True,
        )
    return lane_results


call_model = _recovery_call_model
try:
    for recovery_pass in range(1, MAX_RECOVERY_ATTEMPTS_PER_TRAJECTORY + 1):
        pending_by_model = current_pending_by_model()
        if not pending_by_model:
            break
        print(
            f"Recovery pass {recovery_pass}: ",
            {name: len(values) for name, values in sorted(pending_by_model.items())},
        )
        with ThreadPoolExecutor(max_workers=len(pending_by_model)) as pool:
            futures = {
                pool.submit(run_recovery_lane, name, problems): name
                for name, problems in pending_by_model.items()
            }
            for future in as_completed(futures):
                lane_results = future.result()
                total_attempts_this_run += len(lane_results)
                tqdm.write(
                    f"[{futures[future]}] pass complete: "
                    f"{sum(item[1] for item in lane_results)}/{len(lane_results)} accepted"
                )
        remaining = len(required_keys - set(successful_replacements))
        print(f"Remaining invalid trajectories after pass {recovery_pass}: {remaining}")
        if remaining and recovery_pass < MAX_RECOVERY_ATTEMPTS_PER_TRAJECTORY:
            time.sleep(BETWEEN_RECOVERY_PASSES_SEC)
finally:
    call_model = _base_call_model
    success_progress.close()

unresolved_keys = sorted(required_keys - set(successful_replacements))
print("Successful replacements:", len(successful_replacements))
print("Unresolved after configured attempts:", len(unresolved_keys))
if unresolved_keys:
    display(pd.DataFrame([
        {
            "agent_name": key[0],
            "problem_id": key[1],
            "attempts": attempt_counts[key],
            "original_stop_reason": source_by_key[key].stop_reason,
        }
        for key in unresolved_keys
    ]))
    print(
        "Increase MAX_RECOVERY_ATTEMPTS_PER_TRAJECTORY, then rerun this cell. "
        "Accepted checkpoint trajectories will not be repeated."
    )


## 9. Merge accepted replacements, export, and enforce the 1,800-record gate


In [ ]:
merged_by_key = dict(source_by_key)
merged_by_key.update(successful_replacements)
if len(merged_by_key) != 1800:
    raise RuntimeError(f"Merged key count changed: {len(merged_by_key)}")

remaining_invalid = {
    key: trajectory_validation_reasons(trajectory)
    for key, trajectory in merged_by_key.items()
    if not is_publishable_trajectory(trajectory)
}
model_order = {spec.name: index for index, spec in enumerate(MODEL_SPECS)}
problem_order = {problem["id"]: index for index, problem in enumerate(source_problems)}
merged_records = sorted(
    [trajectory.to_dict() for trajectory in merged_by_key.values()],
    key=lambda item: (
        problem_order[item["problem_id"]],
        model_order[item["agent_name"]],
    ),
)

health_rows = []
for model_name in sorted(model_order, key=model_order.get):
    rows = [item for item in merged_by_key.values() if item.agent_name == model_name]
    stops = pd.Series([item.stop_reason for item in rows]).value_counts().to_dict()
    health_rows.append({
        "agent": model_name,
        "records": len(rows),
        "publishable": sum(is_publishable_trajectory(item) for item in rows),
        "publishable_rate": np.mean([is_publishable_trajectory(item) for item in rows]),
        "remaining_invalid": sum(not is_publishable_trajectory(item) for item in rows),
        "stop_reasons": json.dumps({str(k): int(v) for k, v in stops.items()}, sort_keys=True),
    })
recovery_health = pd.DataFrame(health_rows)
display(recovery_health)

export_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
export_name = f"{RECOVERY_RUN_NAME}_{export_timestamp}"
export_dir = Path(recovery_cfg.output_dir) / export_name
export_dir.mkdir(parents=True, exist_ok=False)


def write_jsonl(path, records):
    with Path(path).open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


write_jsonl(export_dir / "trajectories.jsonl", merged_records)
write_jsonl(export_dir / "problems.jsonl", source_problems)
recovery_health.to_csv(export_dir / "generation_health.csv", index=False)
required_reruns_df.to_csv(export_dir / "required_reruns.csv", index=False)

unresolved_frame = pd.DataFrame([
    {
        "agent_name": key[0],
        "problem_id": key[1],
        "attempts": attempt_counts[key],
        "original_stop_reason": source_by_key[key].stop_reason,
        "remaining_reasons": "|".join(reasons),
    }
    for key, reasons in sorted(remaining_invalid.items())
])
unresolved_frame.to_csv(export_dir / "unresolved_after_recovery.csv", index=False)

checkpoint_copy = export_dir / "recovery_attempts.jsonl"
if recovery_checkpoint.path.exists():
    shutil.copy2(recovery_checkpoint.path, checkpoint_copy)
else:
    checkpoint_copy.write_text("", encoding="utf-8")

recovery_audit = {
    "source_run": source_manifest.get("run_name"),
    "source_trajectory_count": len(source_by_key),
    "selection_predicate": (
        "finished and not timed_out and stop_reason=explicit_final_answer and "
        "nonempty final_answer and steps and answer_step"
    ),
    "rerun_protocol_failures": RERUN_PROTOCOL_FAILURES,
    "selected_for_rerun": len(required_keys),
    "successful_replacements": len(successful_replacements),
    "remaining_invalid": len(remaining_invalid),
    "attempt_counts": {
        f"{agent}::{problem_id}": int(attempt_counts[(agent, problem_id)])
        for agent, problem_id in sorted(required_keys)
    },
    "nvidia_key_slots": NVIDIA_RECOVERY_KEY_SLOTS,
    "key_values_exported": False,
    "methodology_warning": (
        "Protocol-format recovery is a repaired conditional dataset and must not "
        "replace raw operational completion reporting."
    ),
}
(export_dir / "recovery_audit.json").write_text(
    json.dumps(recovery_audit, indent=2), encoding="utf-8"
)

data_files = [
    "trajectories.jsonl", "problems.jsonl", "generation_health.csv",
    "required_reruns.csv", "unresolved_after_recovery.csv",
    "recovery_attempts.jsonl", "recovery_audit.json",
]
export_manifest = {
    "artifact_type": "fully_recovered_generation",
    "raw_data_schema_version": source_manifest.get("raw_data_schema_version", 1),
    "created_at": datetime.now(timezone.utc).isoformat(),
    "run_name": export_name,
    "source_run": source_manifest.get("run_name"),
    "models": source_manifest.get("models", []),
    "trajectory_count": len(merged_records),
    "problem_count": len(source_problems),
    "publishable_count": len(merged_records) - len(remaining_invalid),
    "data_files": data_files,
    "sha256": {
        name: hashlib.sha256((export_dir / name).read_bytes()).hexdigest()
        for name in data_files
    },
}
(export_dir / "generation_manifest.json").write_text(
    json.dumps(export_manifest, indent=2), encoding="utf-8"
)
recovery_zip = Path(shutil.make_archive(
    str(export_dir), "zip", root_dir=export_dir
))
print("Recovery export:", recovery_zip)
show_download_link(recovery_zip)

PUBLISHABLE_COMPLETE = len(merged_records) == 1800 and not remaining_invalid
print("PUBLISHABLE_COMPLETE =", PUBLISHABLE_COMPLETE)


In [ ]:
# Final publication gate. Do not calculate final paper metrics unless this passes.
if REQUIRE_ALL_PUBLISHABLE and not PUBLISHABLE_COMPLETE:
    raise RuntimeError(
        f"Recovery is incomplete: {len(remaining_invalid)} invalid trajectories remain. "
        "Rerun the recovery engine cell; checkpoints preserve accepted replacements."
    )
assert len(merged_records) == 1800
assert len({(item["agent_name"], item["problem_id"]) for item in merged_records}) == 1800
assert all(is_publishable_trajectory(Trajectory.from_dict(item)) for item in merged_records)
print("FINAL GATE PASSED: exactly 1,800 unique publishable trajectories.")
print("Use this recovery ZIP as the only input to strive-metrics-from-trajectories.ipynb")


## 10. Rerun scope

On the first execution, run the notebook from top to bottom.

If the final gate reports unresolved trajectories, change only
`MAX_RECOVERY_ATTEMPTS_PER_TRAJECTORY` if desired, then rerun:

1. **Recovery engine cell** in Section 8.
2. **Merge/export cell** in Section 9.
3. **Final publication gate** in Section 9.

Do not rerun the source generation, sandbox tests, or already accepted
trajectories. The append-only recovery checkpoint resumes automatically.

After the gate passes, use the recovered ZIP as the sole `TRAJECTORY_INPUTS`
entry in `strive-metrics-from-trajectories.ipynb`. Then pass that notebook's
metrics ZIP to `strive-advanced-metric-analysis.ipynb`.
